# 스티어링 벡터 추출하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/ko/04_extract_vector.ipynb)

이 노트북에서는 PSYCTL의 세 가지 추출 방법을 사용하여 대조 데이터셋에서 스티어링 벡터를 추출하는 과정을 안내합니다:

1. **평균 차이** (`mean_diff`) -- 빠르고 단순한 활성화 차이의 평균
2. **잡음 제거 평균 차이** (`denoised_mean_diff`) -- PCA 기반 잡음 제거로 노이즈 감소
3. **BiPO** (`bipo`) -- DPO 손실을 사용한 그래디언트 기반 최적화 (최고 품질)

**이 노트북에서 다루는 내용:**
1. HuggingFace에서 스티어링 데이터셋을 로드합니다
2. 세 가지 방법으로 벡터를 추출합니다
3. 응답을 나란히 비교합니다
4. 결과를 저장하고 선택적으로 공유합니다

**요구사항:** Colab 무료 티어(T4 GPU). [HuggingFace 토큰](https://huggingface.co/settings/tokens).

**소요 시간:** 약 10분

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate datasets

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## 데이터셋 로드

HuggingFace의 커뮤니티 제공 데이터셋을 사용합니다. [03_generate_dataset.ipynb](./03_generate_dataset.ipynb)에서 생성한 로컬 JSONL 파일을 사용할 수도 있습니다.

In [ ]:
from pathlib import Path
from datasets import load_dataset

# Download a community steering dataset
DATASET_NAME = "CaveduckAI/steer-personality-extroversion-ko"
ds = load_dataset(DATASET_NAME, split="train")

# Save as local JSONL for PSYCTL
dataset_dir = Path("./dataset_demo")
dataset_dir.mkdir(exist_ok=True)
dataset_path = dataset_dir / "data.jsonl"
ds.to_json(str(dataset_path))

print(f"Dataset: {DATASET_NAME}")
print(f"Samples: {len(ds)}")
print(f"Saved to: {dataset_path}")
print(f"\nSample keys: {list(ds[0].keys())}")

## 방법 1: 평균 차이 (가장 빠름)

positive 응답과 neutral 응답 사이의 평균 활성화 차이를 계산합니다. [CAA 논문](https://arxiv.org/abs/2312.06681)에서 제안된 가장 단순하고 빠른 방법입니다.

In [ ]:
from psyctl.core.steering_extractor import SteeringExtractor

SMALL_MODEL = "google/gemma-3-270m-it"
TARGET_LAYER = "model.layers.9.mlp.down_proj"  # Middle layer of 18-layer model

extractor = SteeringExtractor()

output_mean_diff = Path("./vectors/mean_diff.safetensors")
vectors_md = extractor.extract_steering_vector(
    model_name=SMALL_MODEL,
    layers=[TARGET_LAYER],
    dataset_path=dataset_path,
    output_path=output_mean_diff,
    method="mean_diff",
    normalize=True,
)
print(f"Mean diff vector shape: {list(vectors_md.values())[0].shape}")
print(f"Saved to: {output_mean_diff}")

## 방법 2: 잡음 제거 평균 차이 (PCA 기반)

활성화 차이에 PCA 잡음 제거를 적용하여 평균을 구하기 전에 노이즈를 줄입니다. 단순 평균 차이보다 더 강건합니다.

In [ ]:
output_denoised = Path("./vectors/denoised_mean_diff.safetensors")
vectors_dmd = extractor.extract_steering_vector(
    model_name=SMALL_MODEL,
    layers=[TARGET_LAYER],
    dataset_path=dataset_path,
    output_path=output_denoised,
    method="denoised_mean_diff",
    normalize=True,
    variance_threshold=0.95,
)
print(f"Denoised mean diff vector shape: {list(vectors_dmd.values())[0].shape}")
print(f"Saved to: {output_denoised}")

## 방법 3: BiPO (그래디언트 기반, 최고 품질)

[양방향 선호도 최적화(BiPO)](https://arxiv.org/abs/2406.00045)를 사용합니다. DPO 손실을 사용하여 스티어링 벡터를 최적화하는 그래디언트 기반 방법입니다. 느리지만 일반적으로 최고의 조향 품질을 제공합니다.

In [ ]:
output_bipo = Path("./vectors/bipo.safetensors")
vectors_bipo = extractor.extract_steering_vector(
    model_name=SMALL_MODEL,
    layers=[TARGET_LAYER],
    dataset_path=dataset_path,
    output_path=output_bipo,
    method="bipo",
    normalize=True,
    epochs=3,
    lr=1e-3,
)
print(f"BiPO vector shape: {list(vectors_bipo.values())[0].shape}")
print(f"Saved to: {output_bipo}")

## 세 가지 방법 비교

각 벡터를 동일한 프롬프트에 적용하고 조향된 출력을 비교합니다.

In [ ]:
from psyctl.core.steering_applier import SteeringApplier

applier = SteeringApplier()
prompt = "Hello, how are you?"

methods = {
    "mean_diff": output_mean_diff,
    "denoised_mean_diff": output_denoised,
    "bipo": output_bipo,
}

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

# Baseline (no steering)
print("\n--- Baseline ---")
print(applier.apply_steering(
    model_name=SMALL_MODEL,
    steering_vector_path=output_mean_diff,
    input_text=prompt,
    strength=0.0,
    max_new_tokens=100,
    temperature=0.7,
))

for method_name, vpath in methods.items():
    print(f"\n--- {method_name} (strength=1.5) ---")
    print(applier.apply_steering(
        model_name=SMALL_MODEL,
        steering_vector_path=vpath,
        input_text=prompt,
        strength=1.5,
        max_new_tokens=100,
        temperature=0.7,
    ))

## 저장 및 공유

추출된 벡터는 `./vectors/`에 `.safetensors` 파일로 저장됩니다. HuggingFace Hub에 업로드하여 커뮤니티와 공유할 수 있습니다:

```python
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="./vectors/bipo.safetensors",
    path_in_repo="my_vector.safetensors",
    repo_id="your-username/my-steering-vectors",
    repo_type="model",
)
```

**다음 단계:** [05_layer_analysis.ipynb](./05_layer_analysis.ipynb)에서 추출에 최적인 레이어를 찾아보세요.